# OpenPath BKT fitting

Fit standard Bayesian Knowledge Tracing separately for each of the ten mapped skills. The model has initial mastery, learning, guess, and slip probabilities, with forgetting fixed at zero. The notebook uses NumPy and SciPy to maximize the likelihood of the observed response histories.

Run the cells in order from the folder containing `bkt_fitting.csv`. The fitting file caps each skill at 30,000 encounters. It preserves Junyi's recorded labels, including the hint override. The fitted parameters describe the mapped Junyi tasks and should be evaluated on OpenPath's own questions before drawing conclusions about mastery there.

[Corbett and Anderson, Knowledge Tracing](https://doi.org/10.1007/BF01099821)

## 1. Set the file paths and fitting options

In [1]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json

from IPython.display import display
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize

DATA_DIR = Path.cwd()
TRAIN_CSV = DATA_DIR / "bkt_fitting.csv"
VALIDATION_SOURCE = Path.home() / "Downloads" / "pathgen0.4" / "csv" / "bkt_fitting.csv"
PARAMETERS_FILE = DATA_DIR / "bkt_parameters.json"
SEED = 42
NUM_STARTS = 8
ROW_CAP = 30_000
EPS = 1e-5
COLUMNS = ["user_id", "order_id", "modeled_skill_id", "correct"]
SKILLS = [
    "INT_NUMBER_LINE", "INT_OPPOSITE_POS4", "INT_ABS_COMPARE", "INT_ORDER_NEG",
    "INT_ADD_NEG", "INT_SUB_NEG", "INT_MULT_NEG", "INT_DIV_NEG", "INT_EXPR_NEG",
    "INT_CONTEXT",
]

assert TRAIN_CSV.is_file(), "Run the notebook from the folder containing bkt_fitting.csv."
print(f"NumPy {np.__version__}, pandas {pd.__version__}, SciPy {scipy.__version__}")
print(f"Training CSV: {TRAIN_CSV.name}")
print(f"Validation source available: {VALIDATION_SOURCE.is_file()}")

NumPy 2.5.3, pandas 3.0.6, SciPy 1.18.1
Training CSV: bkt_fitting.csv
Validation source available: True


## 2. Load and verify the fitting data

In [2]:
def load_responses(path):
    data = pd.read_csv(path, dtype={"user_id": str, "modeled_skill_id": str})
    if list(data.columns) != COLUMNS or data.isna().any().any():
        raise ValueError(f"Unexpected schema or missing values in {path.name}")
    if not data["correct"].isin([0, 1]).all():
        raise ValueError("The correct column must contain only 0 and 1.")
    if not pd.api.types.is_integer_dtype(data["order_id"]) or not data["order_id"].gt(0).all():
        raise ValueError("order_id must contain positive integers.")
    if data.duplicated(["user_id", "order_id"]).any():
        raise ValueError("Duplicate student and order ID.")
    return data.sort_values(["user_id", "order_id"], kind="stable", ignore_index=True)


def student_bucket(prefix, user_id):
    digest = sha256(f"{prefix}{user_id}".encode()).digest()
    return int.from_bytes(digest[:8], "big") < (1 << 64) // 5


train = load_responses(TRAIN_CSV)
training_users = set(train["user_id"])
if any(student_bucket("42|", user) or student_bucket("42|bkt-validation|", user)
       for user in training_users):
    raise ValueError("The fitting CSV contains students reserved for validation or test.")

training_summary = train.groupby("modeled_skill_id").agg(
    rows=("correct", "size"), students=("user_id", "nunique"), correct_rate=("correct", "mean")
)
if set(training_summary.index) != set(SKILLS) or training_summary["rows"].max() > ROW_CAP:
    raise ValueError("Expected the ten mapped skills with at most 30,000 rows each.")
training_summary = training_summary.loc[SKILLS]
print(f"{len(train):,} training rows across {len(training_users):,} students.")
display(training_summary.round(4))

263,813 training rows across 10,114 students.


,rows,students,correct_rate
modeled_skill_id,,,
INT_NUMBER_LINE,29976,1632,0.7430
INT_OPPOSITE_POS4,11863,3195,0.7617
INT_ABS_COMPARE,29983,1860,0.8791
INT_ORDER_NEG,29990,2254,0.9164
INT_ADD_NEG,29999,1424,0.8427
INT_SUB_NEG,29989,2467,0.6671
INT_MULT_NEG,27442,3465,0.8906
INT_DIV_NEG,27536,3502,0.8447
INT_EXPR_NEG,17038,1168,0.5472


## 3. Define and check the BKT likelihood

With no forgetting, a student can move from unmastered to mastered at most once. For each response history, the likelihood sums over all possible points of learning, including already mastered at the start and remaining unmastered through the final answer. The checks compare that calculation with a separate, sequential BKT update and verify the optimizer's gradient.

In [3]:
def likelihood_counts(responses):
    lengths = np.array([len(y) for y in responses], dtype=np.int64)
    starts = np.r_[0, np.cumsum(lengths[:-1] + 1)]
    path_starts = np.repeat(starts, lengths + 1)
    path_lengths = np.repeat(lengths, lengths + 1)
    unmastered_count = np.arange(len(path_starts)) - path_starts
    unmastered_right = np.concatenate([np.r_[0, np.cumsum(y)] for y in responses])
    total_right = np.repeat([y.sum() for y in responses], lengths + 1)
    mastered_right = total_right - unmastered_right
    return (
        starts, np.repeat(np.arange(len(lengths)), lengths + 1), int(lengths.sum()),
        (unmastered_count == 0), (unmastered_count > 0),
        (unmastered_count > 0) & (unmastered_count < path_lengths),
        np.maximum(0, unmastered_count - 1),
        unmastered_right, unmastered_count - unmastered_right,
        mastered_right, path_lengths - unmastered_count - mastered_right,
    )


def objective(theta, counts):
    initial, learn, guess, gain = theta
    mastered_correct = guess + (1 - guess) * gain
    (starts, group, observations, known, unknown, learned, stayed,
     u_right, u_wrong, m_right, m_wrong) = counts
    log_weights = (
        known * np.log(initial) + unknown * np.log1p(-initial)
        + learned * np.log(learn) + stayed * np.log1p(-learn)
        + u_right * np.log(guess) + u_wrong * np.log1p(-guess)
        + m_right * np.log(mastered_correct) + m_wrong * np.log1p(-mastered_correct)
    )
    maxima = np.maximum.reduceat(log_weights, starts)
    shifted = np.exp(log_weights - maxima[group])
    totals = np.add.reduceat(shifted, starts)
    weights = shifted / totals[group]
    m_gradient = np.sum(weights * (m_right / mastered_correct - m_wrong / (1 - mastered_correct)))
    gradient = np.array([
        np.sum(weights * (known / initial - unknown / (1 - initial))),
        np.sum(weights * (learned / learn - stayed / (1 - learn))),
        np.sum(weights * (u_right / guess - u_wrong / (1 - guess))) + (1 - gain) * m_gradient,
        (1 - guess) * m_gradient,
    ])
    return -(maxima + np.log(totals)).sum() / observations, -gradient / observations


def probabilities(theta):
    initial, learn, guess, gain = map(float, theta)
    return {"initial_mastery": initial, "learn": learn, "guess": guess,
            "slip": (1 - guess) * (1 - gain), "forget": 0.0}


def predict_sequence(outcomes, parameters):
    mastery = parameters["initial_mastery"]
    guess = parameters["guess"]
    mastered_correct = 1 - parameters["slip"]
    learn = parameters["learn"]
    predictions = np.empty(len(outcomes), dtype=float)
    for i, correct in enumerate(outcomes):
        p_correct = mastery * mastered_correct + (1 - mastery) * guess
        predictions[i] = p_correct
        if correct:
            posterior = mastery * mastered_correct / p_correct
        else:
            posterior = mastery * (1 - mastered_correct) / (1 - p_correct)
        mastery = posterior + (1 - posterior) * learn
    return predictions


def log_loss(outcomes, predictions):
    p = np.clip(predictions, np.finfo(float).eps, 1 - np.finfo(float).eps)
    return float(-np.where(outcomes == 1, np.log(p), np.log1p(-p)).mean())


def student_sequences(data):
    return [group["correct"].to_numpy(dtype=np.int64) for _, group in data.groupby("user_id", sort=False)]


check = [seq[:12] for seq in student_sequences(train[train["modeled_skill_id"].eq(SKILLS[0])])[:12]]
theta = np.array([0.24, 0.15, 0.31, 0.76])
check_counts = likelihood_counts(check)
loss, gradient = objective(theta, check_counts)
sequential_loss = log_loss(np.concatenate(check), np.concatenate([
    predict_sequence(seq, probabilities(theta)) for seq in check
]))
np.testing.assert_allclose(loss, sequential_loss, rtol=1e-10, atol=1e-10)
step = 1e-6
finite_differences = np.array([
    (objective(theta + direction, check_counts)[0] - objective(theta - direction, check_counts)[0]) / (2 * step)
    for direction in np.eye(4) * step
])
np.testing.assert_allclose(gradient, finite_differences, rtol=1e-4, atol=1e-6)
print("Likelihood and analytic gradient verified on real response histories.")

Likelihood and analytic gradient verified on real response histories.


## 4. Fit each skill separately

Use eight seeded starting points per skill and keep the converged solution with the lowest training log loss. The optimizer constrains the probability of a correct response to be higher when mastered. The validation data is not used to choose a fit.

In [4]:
def fit_skill(responses, seed):
    counts = likelihood_counts(responses)
    rng = np.random.default_rng(seed)
    starts = [np.array([0.20, 0.15, 0.25, 0.75])]
    starts.extend(rng.uniform([0.05, 0.03, 0.08, 0.35], [0.85, 0.55, 0.70, 0.96],
                              size=(NUM_STARTS - 1, 4)))
    bounds = [(EPS, 1 - EPS), (EPS, 1 - EPS), (EPS, 1 - 2 * EPS), (EPS, 1 - EPS)]
    fits = [minimize(objective, start, args=(counts,), jac=True, method="L-BFGS-B",
                     bounds=bounds, options={"maxiter": 450, "ftol": 1e-11, "gtol": 1e-7})
            for start in starts]
    converged = [fit for fit in fits if fit.success and np.isfinite(fit.fun)]
    if not converged:
        raise RuntimeError("All optimization starts failed to converge.")
    best = min(converged, key=lambda fit: fit.fun)
    parameters = probabilities(best.x)
    if not 0 < parameters["guess"] < 1 - parameters["slip"] < 1:
        raise ValueError("Invalid emission probabilities.")
    boundary = [name for name in ("initial_mastery", "learn", "guess", "slip")
                if parameters[name] < 1e-3 or parameters[name] > 1 - 1e-3]
    return parameters, float(best.fun), len(converged), boundary


fitted = {}
for index, skill in enumerate(SKILLS):
    began = perf_counter()
    skill_rows = train[train["modeled_skill_id"].eq(skill)]
    parameters, loss, converged, boundary = fit_skill(student_sequences(skill_rows), SEED + index)
    fitted[skill] = {
        **parameters, "training_rows": int(len(skill_rows)),
        "training_students": int(training_summary.loc[skill, "students"]),
        "training_log_loss": loss, "converged_starts": converged,
        "near_boundary_parameters": boundary,
    }
    print(f"{skill}: {len(skill_rows):,} rows, {converged}/{NUM_STARTS} converged, "
          f"log loss {loss:.4f}, {perf_counter() - began:.2f}s", flush=True)

display(pd.DataFrame.from_dict(fitted, orient="index").round(5))

INT_NUMBER_LINE: 29,976 rows, 8/8 converged, log loss 0.5242, 0.19s
INT_OPPOSITE_POS4: 11,863 rows, 8/8 converged, log loss 0.4551, 0.22s
INT_ABS_COMPARE: 29,983 rows, 8/8 converged, log loss 0.3330, 0.30s
INT_ORDER_NEG: 29,990 rows, 8/8 converged, log loss 0.2673, 0.28s
INT_ADD_NEG: 29,999 rows, 8/8 converged, log loss 0.4029, 0.22s
INT_SUB_NEG: 29,989 rows, 8/8 converged, log loss 0.5685, 0.28s
INT_MULT_NEG: 27,442 rows, 8/8 converged, log loss 0.3279, 0.28s
INT_DIV_NEG: 27,536 rows, 8/8 converged, log loss 0.4147, 0.26s
INT_EXPR_NEG: 17,038 rows, 8/8 converged, log loss 0.5626, 0.13s
INT_CONTEXT: 29,997 rows, 8/8 converged, log loss 0.5131, 0.25s


,initial_mastery,learn,guess,slip,forget,training_rows,training_students,training_log_loss,converged_starts,near_boundary_parameters
INT_NUMBER_LINE,0.35480,0.08322,0.42145,0.14739,0.0,29976,1632,0.52415,8,[]
INT_OPPOSITE_POS4,0.50369,0.88630,0.00001,0.10884,0.0,11863,3195,0.45507,8,[guess]
INT_ABS_COMPARE,0.79838,0.01199,0.70958,0.03927,0.0,29983,1860,0.33298,8,[]
INT_ORDER_NEG,0.60290,0.79034,0.30084,0.06011,0.0,29990,2254,0.26734,8,[]
INT_ADD_NEG,0.70990,0.01067,0.67107,0.06684,0.0,29999,1424,0.40294,8,[]
INT_SUB_NEG,0.65602,0.02143,0.42843,0.11572,0.0,29989,2467,0.56850,8,[]
INT_MULT_NEG,0.79604,0.09402,0.61494,0.06010,0.0,27442,3465,0.32790,8,[]
INT_DIV_NEG,0.76223,0.08246,0.57292,0.09194,0.0,27536,3502,0.41467,8,[]
INT_EXPR_NEG,0.34243,0.03333,0.15548,0.24221,0.0,17038,1168,0.56262,8,[]
INT_CONTEXT,0.36015,0.09065,0.35946,0.14702,0.0,29997,1950,0.51308,8,[]


## 5. Evaluate on held-out Junyi validation students

The larger source file provides the learners previously reserved by the validation hash rule. For each response, score the prediction before using the result to update mastery. Compare with a constant baseline equal to that skill's success rate in the capped fitting data. Lower log loss and Brier scores are better. The separate Junyi test partition stays reserved for later evaluation.

In [5]:
validation_results = {}
validation_students = 0
validation_rows = 0
if VALIDATION_SOURCE.is_file():
    development = load_responses(VALIDATION_SOURCE)
    development_users = set(development["user_id"])
    reserved_users = {user for user in development_users if student_bucket("42|bkt-validation|", user)}
    if training_users & reserved_users or any(student_bucket("42|", user) for user in reserved_users):
        raise ValueError("Validation contains students from the training or test split.")
    validation = development[development["user_id"].isin(reserved_users)]
    del development
    validation_students = len(reserved_users)
    validation_rows = len(validation)
    if set(validation["modeled_skill_id"]) != set(SKILLS):
        raise ValueError("Validation data must cover all ten skills.")
    for skill in SKILLS:
        skill_rows = validation[validation["modeled_skill_id"].eq(skill)]
        responses = student_sequences(skill_rows)
        outcomes = np.concatenate(responses)
        predictions = np.concatenate([predict_sequence(y, fitted[skill]) for y in responses])
        baseline = float(training_summary.loc[skill, "correct_rate"])
        validation_results[skill] = {
            "rows": int(len(outcomes)), "students": int(len(responses)),
            "bkt_log_loss": log_loss(outcomes, predictions),
            "baseline_log_loss": log_loss(outcomes, np.full(len(outcomes), baseline)),
            "bkt_brier": float(np.mean((outcomes - predictions) ** 2)),
            "baseline_brier": float(np.mean((outcomes - baseline) ** 2)),
        }
    table = pd.DataFrame.from_dict(validation_results, orient="index")
    improvement = int((table["bkt_log_loss"] < table["baseline_log_loss"]).sum())
    print(f"{validation_rows:,} validation rows from {validation_students:,} held-out students.")
    print(f"BKT improved log loss for {improvement}/{len(SKILLS)} skills.")
    print(f"Mean across skills: BKT {table['bkt_log_loss'].mean():.4f}, "
          f"baseline {table['baseline_log_loss'].mean():.4f}.")
    display(table.round(4))
else:
    print("Validation source unavailable. Set VALIDATION_SOURCE and rerun this cell to score the fit.")

152,659 validation rows from 3,921 held-out students.
BKT improved log loss for 10/10 skills.
Mean across skills: BKT 0.4457, baseline 0.4936.


,rows,students,bkt_log_loss,baseline_log_loss,bkt_brier,baseline_brier
INT_NUMBER_LINE,37124,2042,0.5235,0.5693,0.1723,0.1907
INT_OPPOSITE_POS4,2969,805,0.4516,0.5563,0.1435,0.1848
INT_ABS_COMPARE,12473,740,0.3792,0.4126,0.1148,0.1231
INT_ORDER_NEG,22835,1800,0.2653,0.2888,0.0717,0.0771
INT_ADD_NEG,34645,1541,0.4127,0.4428,0.1267,0.1357
INT_SUB_NEG,14222,1176,0.5846,0.6423,0.1994,0.2250
INT_MULT_NEG,6300,818,0.3205,0.3302,0.0898,0.0918
INT_DIV_NEG,6283,832,0.4210,0.4354,0.1283,0.1326
INT_EXPR_NEG,3666,281,0.5743,0.6857,0.1939,0.2463
INT_CONTEXT,12142,808,0.5241,0.5723,0.1727,0.1920


## 6. Export and verify the parameters

The JSON contains the shared fit for each skill and the validation metrics. When creating agent training features, fit separate BKT models on the other training folds for each learner. Use this frozen fit for held-out learners and update their mastery from earlier responses.

In [6]:
payload = {
    "schema_version": 1,
    "model": "four_parameter_bkt_no_forgetting",
    "fitter": "exact_sequence_likelihood_scipy_lbfgsb",
    "observation": "Junyi recorded correct flag, including the hint override",
    "training_file": TRAIN_CSV.name,
    "training_sha256": sha256(TRAIN_CSV.read_bytes()).hexdigest(),
    "training_rows": int(len(train)),
    "training_students": len(training_users),
    "max_rows_per_skill": ROW_CAP,
    "seed": SEED,
    "starts_per_skill": NUM_STARTS,
    "validation_rows": validation_rows,
    "validation_students": validation_students,
    "skills": fitted,
    "validation": validation_results,
}
assert set(fitted) == set(SKILLS)
assert sum(item["training_rows"] for item in fitted.values()) == len(train)
serialized = json.dumps(payload, indent=2, allow_nan=False) + "\n"
temporary = PARAMETERS_FILE.with_suffix(".json.tmp")
temporary.write_text(serialized, encoding="utf-8")
temporary.replace(PARAMETERS_FILE)
assert json.loads(PARAMETERS_FILE.read_text(encoding="utf-8")) == payload
print(f"Wrote and verified {PARAMETERS_FILE.name} with parameters for {len(fitted)} skills.")

Wrote and verified bkt_parameters.json with parameters for 10 skills.
